## Pitch Visualization

In [133]:
!uv pip install scikit-learn


Using Python 3.14.3 environment at: /Users/charlotteschwegler/Desktop/GDV_Repository/Praediktive-Direkte-Demokratie/.venv
Audited 1 package in 17ms


In [134]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Datenset laden

In [135]:
data = pd.read_csv('../data/DATASET CSV 11-02-2026.csv', sep=';')
data['br-pos'] = pd.to_numeric(data['br-pos'], errors='coerce')
data['annahme'] = pd.to_numeric(data['annahme'], errors='coerce')

# Überblick über die Daten schaffen

In [136]:
data.head()
data.shape
print(data.columns)
data.info()
data.isna().sum()
unique_titel = data['titel_kurz_d'].unique()
len(unique_titel)
unique_titel[:20]

Index(['anr', 'datum', 'titel_kurz_d', 'titel_kurz_f', 'titel_kurz_e',
       'titel_off_d', 'titel_off_f', 'stichwort', 'swissvoteslink', 'anzahl',
       ...
       'bkresults-fr', 'bfsdash-de', 'bfsdash-fr', 'bfsdash-en', 'bfsmap-de',
       'bfsmap-fr', 'bfsmap-en', 'nach_cockpit_d', 'nach_cockpit_f',
       'nach_cockpit_e'],
      dtype='str', length=874)
<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Columns: 874 entries, anr to nach_cockpit_e
dtypes: float64(482), int64(10), str(382)
memory usage: 4.7 MB


<StringArray>
[                     'Bundesverfassung der schweizerischen Eidgenossenschaft',
                                                            'Mass und Gewicht',
    'Gleichstellung der Juden und Naturalisierten mit Bezug auf Niederlassung',
                  'Stimmrecht der Niedergelassenen in Gemeindeangelegenheiten',
           'Besteuerung und zivilrechtliche Verhältnisse der Niedergelassenen',
               'Stimmrecht der Niedergelassenen in kantonalen Angelegenheiten',
                                                'Glaubens- und Kultusfreiheit',
                                         'Ausschliessung einzelner Strafarten',
                                              'Schutz des geistigen Eigentums',
                                        'Verbot der Lotterie und Hasardspiele',
                                            'Bundesverfassung (Totalrevision)',
 'Gesetz betreffend Feststellung und Beurkundung des Zivilstandes und die Ehe',
            'Gesetz über d

# Wrangling und Datacleaning

In [137]:
data['datum'] = pd.to_datetime(data['datum'], format='%d.%m.%Y')

data.replace([9999,999], np.nan, inplace=True)

# Kongruenz-Spalte erstellen (Regierungstreue)
#br-pos: 1 = Ja, 2 = Nein, 8 = Vorzug Gegenentwurf | Volksmehr: 1 = Angenommen, 0 = Abgelehnt, 8 = Vorzug Gegenentwurf

data['regierungstreue'] = np.where(
     ((data['br-pos'] == 1) & (data['volk'] == 1)) |
     ((data['br-pos'] == 2) & (data['volk'] == 0)) |
     ((data['br-pos'] == 8) & (data['volk'] == 8)) |
    ((data['br-pos'] == 9) & (data['volk'] == 9)),
     1,
     0
 )


data['regierungstreue'] = np.where(
    data['br-pos'].isin([1, 2]),
    np.where(
        ((data['br-pos'] == 1) & (data["volkja-proz"] > 50.0)) |
        ((data['br-pos'] == 2) & (data["volkja-proz"] < 50.0)),
        1,
        0
    ),
    np.nan
)
for col in ['d1e1', 'd1e2', 'd1e3', 'd2e1', 'd2e2', 'd2e3', 'd3e1', 'd3e2', 'd3e3']:
    print(f"\n{col}")
    print(data[col].value_counts(dropna=False).head(20))
    
hauptgruppen_map = {
    1: 'Staatsordnung & Politische Rechte',
    2: 'EU & Aussenpolitik',
    3: 'Armee & Sicherheit',
    4: 'Wirtschaft & Arbeit',
    5: 'Landwirtschaft',
    6: 'Finanzen & Steuern',
    7: 'Energie',
    8: 'Verkehr & Infrastruktur',
    9: 'Klima & Umwelt',
    10: 'Soziale Sicherheit & Gesellschaft',
    11: 'Bildung & Forschung',
    12: 'Kultur, Religion & Medien'
}

data['Hauptgruppe'] = data['d1e1'].map(hauptgruppen_map)


def assign_hauptgruppe(row):
    d1 = row['d1e1']
    d2 = row['d1e2']
    d3 = row['d1e3']

    if d1 == 10:
        if d2 == 1:
            return 'Gesundheit'
        elif d2 == 2:
            return 'Soziale Sicherheit'
        elif d2 == 3:
            return 'Migration & Gesellschaft'
        else:
            return 'Soziale Sicherheit & Gesellschaft'

    elif d1 == 11:
        return 'Bildung & Forschung'
    elif d1 == 12:
        return 'Kultur, Religion & Medien'
    elif d1 == 2:
        return 'EU & Aussenpolitik'
    elif d1 == 3:
        return 'Armee & Sicherheit'
    elif d1 in [7, 9]:
        return 'Klima & Umwelt'
    elif d1 == 6:
        return 'Finanzen & Steuern'
    elif d1 == 8:
        return 'Verkehr & Infrastruktur'
    elif d1 == 4:
        return 'Wirtschaft & Arbeit'
    elif d1 == 5:
        return 'Landwirtschaft'
    elif d1 == 1:
        return 'Staatsordnung & Politische Rechte'
    else:
        return 'Andere'
    
data['Hauptgruppe'] = data.apply(assign_hauptgruppe, axis=1)

data['Hauptgruppe'].value_counts(dropna=False)
data[['titel_kurz_d', 'd1e1', 'd1e2', 'd1e3', 'Hauptgruppe']].sample(20, random_state=42)
data[data['Hauptgruppe'].isna()][['titel_kurz_d', 'd1e1', 'd1e2', 'd1e3']].head(20)

# Labels für die Spalte Rechtsform vergeben
rechtsform_map = {
    1: 'Obligatorisches Referendum',
    2: 'Fakultatives Referendum',
    3: 'Volksinitiative',
    4: 'Direkter Gegenentwurf',
    5: 'Stichfrage'
}

data['rechtsform_name'] = data['rechtsform'].map(rechtsform_map)

# Abstimmungen in den Kantonen: Spalten ordnen
kanton_cols = [col for col in data.columns if col.endswith('-japroz')]
print(kanton_cols)
print(len(kanton_cols))

kanton_df = data.melt(
    id_vars=['anr', 'titel_kurz_d', 'regierungstreue', 'br-pos', 'annahme'],
    value_vars=kanton_cols,
    var_name='Kanton',
    value_name='Ja_Anteil'
)

kanton_df['Kanton'] = kanton_df['Kanton'].str.replace('-japroz', '', regex=False).str.upper()

# Jahre umwandeln
data['jahr'] = data['datum'].dt.year

# Vorbereitung für Visualisierungen
plot_data = data[
    data['br-pos'].isin([1, 2]) &
    data['annahme'].isin([0, 1])
].copy()

### CHARLOTTE: WÜRDE für die meisten Analysen LEGISJAHR NEHMEN --> DEKADE LÖSCHEN
plot_data['jahr'] = plot_data['datum'].dt.year
plot_data['dekade'] = (plot_data['jahr'] // 10) * 10

### CHARLOTTE: WÜRDE ICH NICHT SO MACHEN. GROSSER KONZEPTIONELLER UNTERSCHIED ZWISCHEN DEN BEIDEN.
##plot_data['rechtsform_group'] = plot_data['rechtsform_name'].replace({
#   'Obligatorisches Referendum': 'Referendum',
#    'Fakultatives Referendum': 'Referendum'
#})

# Kantonale Annahme-Spalten finden
kanton_annahme_cols = [col for col in data.columns if col.endswith('-annahme')]

# In Long-Format bringen
kanton_vote_df = data.melt(
    id_vars=['anr', 'titel_kurz_d', 'br-pos'],
    value_vars=kanton_annahme_cols,
    var_name='Kanton',
    value_name='kanton_annahme'
)

# Kantonskürzel bereinigen
kanton_vote_df['Kanton'] = kanton_vote_df['Kanton'].str.replace('-annahme', '', regex=False).str.upper()

# Nur klare Fälle behalten
kanton_vote_df = kanton_vote_df[
    kanton_vote_df['br-pos'].isin([1, 2]) &
    kanton_vote_df['kanton_annahme'].isin([0, 1])
].copy()

# Kantonale Regierungstreue berechnen
kanton_vote_df['kanton_regierungstreue'] = np.where(
    ((kanton_vote_df['br-pos'] == 1) & (kanton_vote_df['kanton_annahme'] == 1)) |
    ((kanton_vote_df['br-pos'] == 2) & (kanton_vote_df['kanton_annahme'] == 0)),
    1,
    0
)

# Sprachregionen zuordnen
sprachregion_map = {
    'ZH': 'Deutschschweiz', 'BE': 'Deutschschweiz', 'LU': 'Deutschschweiz',
    'UR': 'Deutschschweiz', 'SZ': 'Deutschschweiz', 'OW': 'Deutschschweiz',
    'NW': 'Deutschschweiz', 'GL': 'Deutschschweiz', 'ZG': 'Deutschschweiz',
    'FR': 'Romandie', 'SO': 'Deutschschweiz', 'BS': 'Deutschschweiz',
    'BL': 'Deutschschweiz', 'SH': 'Deutschschweiz', 'AR': 'Deutschschweiz',
    'AI': 'Deutschschweiz', 'SG': 'Deutschschweiz', 'GR': 'Deutschschweiz',
    'AG': 'Deutschschweiz', 'TG': 'Deutschschweiz', 'TI': 'Tessin',
    'VD': 'Romandie', 'VS': 'Romandie', 'NE': 'Romandie',
    'GE': 'Romandie', 'JU': 'Romandie'
}

# Pro Kanton mitteln
kanton_summary = (
    kanton_vote_df.groupby('Kanton', as_index=False)['kanton_regierungstreue']
    .mean()
)

kanton_summary['Sprachregion'] = kanton_summary['Kanton'].map(sprachregion_map)

# Nach Regierungstreue sortieren
kanton_summary = kanton_summary.sort_values('kanton_regierungstreue')

# Kantons-Spalten
kanton_cols = [col for col in data.columns if col.endswith('-japroz')]

# Long-Format: jede Abstimmung x jeder Kanton
heatmap_df = data.melt(
    id_vars=['anr', 'titel_kurz_d', 'Hauptgruppe'],
    value_vars=kanton_cols,
    var_name='Kanton',
    value_name='Ja_Anteil'
)

# Kantonsnamen bereinigen
heatmap_df['Kanton'] = heatmap_df['Kanton'].str.replace('-japroz', '', regex=False).str.upper()

# Ja-Anteil numerisch
heatmap_df['Ja_Anteil'] = pd.to_numeric(heatmap_df['Ja_Anteil'], errors='coerce')

# Fehlende Werte entfernen
heatmap_df = heatmap_df.dropna(subset=['Ja_Anteil', 'Hauptgruppe'])

# Durchschnittlicher Ja-Anteil pro Hauptgruppe und Kanton
heatmap_data = (
    heatmap_df.groupby(['Hauptgruppe', 'Kanton'])['Ja_Anteil']
    .mean()
    .reset_index()
    .pivot(index='Hauptgruppe', columns='Kanton', values='Ja_Anteil')
)


d1e1
d1e1
10    142
1     107
4      86
6      71
8      58
3      53
9      49
5      38
2      34
11     25
7      24
12     21
Name: count, dtype: int64

d1e2
d1e2
10.2    61
4.1     50
6.1     42
10.3    42
3.2     40
10.1    39
1.6     35
1.3     32
8.2     29
1.4     21
4.2     19
9.3     19
9.2     18
4.3     17
5.3     16
6.2     14
3.1     12
9.1     12
2.2     12
1.2     11
Name: count, dtype: int64

d1e3
d1e3
.        194
10.21     25
10.31     21
10.14     19
10.24     19
6.13      17
4.31      15
1.65      14
4.13      14
3.23      14
1.31      13
6.14      13
10.11     11
1.32      11
4.14      10
9.21      10
2.22      10
4.11       9
8.21       9
1.43       8
Name: count, dtype: int64

d2e1
d2e1
.      213
1      128
10      85
6       84
4       57
9       52
3       24
5       14
2       14
11      10
12       9
7        9
8        6
NaN      3
Name: count, dtype: int64

d2e2
d2e2
.       222
6.1      63
10.3     52
1.6      48
9.3      44
1.5      33
10.2     23
1.2

# Build Index

In [138]:
# Positionen Bundesrat
data['br-pos_label'] = data['br-pos'].map({
    1: 'Befürwortend',
    2: 'Ablehnend',
    3: 'Keine Empfehlung',
    8: "Stichfragen: Vorzug Gegenentwurf",
    9: "Stichfragen: Vorzug Volksinitiative"
})
data['br-pos_label'] = data['br-pos_label'].fillna("Missing")
data['br-pos_label'].value_counts()


br-pos_label
Befürwortend                        333
Ablehnend                           235
Missing                             129
Keine Empfehlung                      7
Stichfragen: Vorzug Gegenentwurf      4
Name: count, dtype: int64

In [139]:
# Positionen Parlament
data['bv-pos_label'] = data['bv-pos'].map({
    1: 'Befürwortend',
    2: 'Ablehnend',
    3: 'Keine Abstimmungsempfehlung',
    8: "Stichfragen: Vorzug Gegenentwurf",
    9: "Stichfragen: Vorzug Volksinitiative"
})
data['bv-pos_label'] = data['bv-pos_label'].fillna("Missing")
data['bv-pos_label'].value_counts()

bv-pos_label
Befürwortend                        466
Ablehnend                           231
Keine Abstimmungsempfehlung           7
Stichfragen: Vorzug Gegenentwurf      4
Name: count, dtype: int64

In [140]:
data["volkja-proz"].value_counts()
data["bet"].value_counts()




bet
.        19
60.09     4
54.33     4
37.85     4
46.95     3
         ..
45.04     1
44.90     1
44.87     1
49.50     1
42.95     1
Name: count, Length: 553, dtype: int64

In [141]:
df_index = data.loc[(data["jahr"] > 1969) & (data["jahr"] < 2026), ["br-pos",  "volkja-proz", "bet", "jahr", "titel_kurz_d"]]
# 1. Erstelle ein Mapping für die Richtung (Multiplier)


print(df_index)

     br-pos  volkja-proz    bet  jahr  \
223     1.0        54.24  43.76  1970   
224     2.0        45.99  74.72  1970   
225     NaN        74.63  43.77  1970   
226     2.0        48.92  43.81  1970   
227     NaN        55.38  40.95  1970   
..      ...          ...    ...   ...   
695     2.0        30.16  37.94  2025   
696     1.0        57.73  49.50  2025   
697     1.0        50.39  49.55  2025   
698     2.0        15.85  42.94  2025   
699     2.0        21.72  42.95  2025   

                                          titel_kurz_d  
223  Bundesbeschluss über die inländische Zuckerwir...  
224                Initiative «Gegen die Überfremdung»  
225  Verfassungsartikel zur Förderung von Turnen un...  
226  Initiative «Recht auf Wohnung und Ausbau des F...  
227                         Änderung der Finanzordnung  
..                                                 ...  
695                   «Umweltverantwortungsinitiative»  
696      Kantonale Besteuerung von Zweitliegenschaf

# Datenbereinigung

In [142]:
df_index[["bet", 'volkja-proz']] = df_index[["bet", 'volkja-proz']].astype(float) / 100
print(df_index)

     br-pos  volkja-proz     bet  jahr  \
223     1.0       0.5424  0.4376  1970   
224     2.0       0.4599  0.7472  1970   
225     NaN       0.7463  0.4377  1970   
226     2.0       0.4892  0.4381  1970   
227     NaN       0.5538  0.4095  1970   
..      ...          ...     ...   ...   
695     2.0       0.3016  0.3794  2025   
696     1.0       0.5773  0.4950  2025   
697     1.0       0.5039  0.4955  2025   
698     2.0       0.1585  0.4294  2025   
699     2.0       0.2172  0.4295  2025   

                                          titel_kurz_d  
223  Bundesbeschluss über die inländische Zuckerwir...  
224                Initiative «Gegen die Überfremdung»  
225  Verfassungsartikel zur Förderung von Turnen un...  
226  Initiative «Recht auf Wohnung und Ausbau des F...  
227                         Änderung der Finanzordnung  
..                                                 ...  
695                   «Umweltverantwortungsinitiative»  
696      Kantonale Besteuerung von Zwei

In [143]:
from sklearn.preprocessing import StandardScaler

In [144]:
# 1. PCA initialisieren und auf die skalierten Daten anwenden
pca = PCA(n_components=1)
df_index['pca_index'] = pca.fit_transform(df_index[["bet", "volkja-proz"]])

# 2. Mapping für die Bundesratsposition (br-pos) definieren
richtung_map = {
    1: 1,   # Befürwortend
    2: -1,  # Ablehnend
    3: 1,   # Keine (neutral behandelt)
    8: -1,  # Vorzug Gegenentwurf
    9: 1    # Vorzug Initiative
}

# 3. Multiplikator zuweisen und finalen Index berechnen
df_index['multiplikator'] = df_index['br-pos'].map(richtung_map)
df_index['regierungs_treue'] = df_index['pca_index'] * df_index['multiplikator']

# --- Ausgabe der Ergebnisse ---
print("Erste Zeilen der Berechnung:")
print(df_index[['br-pos', 'pca_index', 'regierungs_treue']].head())

# --- Analyse der Gewichtung ---
loadings = pca.components_
print(f"\nInterne Gewichtung der PCA:")
print(f"Gewichtung Beteiligung: {loadings[0][0]:.2f}")
print(f"Gewichtung Ja-Anteil:   {loadings[0][1]:.2f}")


Erste Zeilen der Berechnung:
     br-pos  pca_index  regierungs_treue
223     1.0   0.044148          0.044148
224     2.0  -0.071634          0.071634
225     NaN   0.246820               NaN
226     2.0  -0.008789          0.008789
227     NaN   0.058545               NaN

Interne Gewichtung der PCA:
Gewichtung Beteiligung: -0.11
Gewichtung Ja-Anteil:   0.99
